<a href="https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks%20/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlyRank-Internship/week1-flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline Rule

Pages should be reviewed first if they receive high search visibility but have a low click-through rate and have not been updated for a long time. These pages may benefit from refreshing content or improving titles and meta descriptions to increase clicks.

### Reason Codes

**high_impressions_low_ctr** – The page receives many impressions but has a low CTR.

**stale_content** – The page has not been updated for a long time.

**high_priority_refresh** – The page satisfies both conditions and should be reviewed first.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
df[["impressions_90d", "ctr", "days_since_last_update"]].describe()

,impressions_90d,ctr,days_since_last_update
count,30000.000000,30000.000000,30000.000000
mean,5200.366300,0.510733,46.098300
std,16838.019547,3.279162,42.078709
min,1.000000,0.000000,1.000000
25%,81.000000,0.000000,20.000000
50%,731.000000,0.070000,20.000000
75%,3615.250000,0.290000,104.000000
max,517715.000000,100.000000,373.000000


In [ ]:
import os
import numpy as np

# Thresholds from the dataset
HIGH_IMPRESSIONS = 3615
LOW_CTR = 0.29
STALE_DAYS = 104

# Binary signals
df["high_impressions"] = (df["impressions_90d"] >= HIGH_IMPRESSIONS).astype(int)
df["low_ctr"] = (df["ctr"] < LOW_CTR).astype(int)
df["stale_content"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)

# Transparent baseline score
df["baseline_score"] = (
    df["high_impressions"] * 2 +
    df["low_ctr"] * 2 +
    df["stale_content"] * 1
)

# Reason code
def reason_code(row):
    if row["high_impressions"] and row["low_ctr"] and row["stale_content"]:
        return "high_priority_refresh"
    elif row["high_impressions"] and row["low_ctr"]:
        return "high_impressions_low_ctr"
    elif row["stale_content"]:
        return "stale_content"
    else:
        return "other"

df["reason_code"] = df.apply(reason_code, axis=1)

# Action label
df["action"] = np.where(
    df["baseline_score"] >= 4,
    "Review and Refresh",
    "Monitor"
)

# Rank pages
ranked = (
    df.sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")
print()
print(ranked[
    ["content_id",
     "baseline_score",
     "reason_code",
     "action",
     "impressions_90d",
     "ctr",
     "days_since_last_update"]
].head(10))

CSV written successfully.

             content_id  baseline_score            reason_code  \
0  content_5fe46e04994d               5  high_priority_refresh   
1  content_2dba2b1f9536               5  high_priority_refresh   
2  content_cb112fce36be               5  high_priority_refresh   
3  content_36ff89c8214e               5  high_priority_refresh   
4  content_b28d1efd668f               5  high_priority_refresh   
5  content_813e88069237               5  high_priority_refresh   
6  content_c8e9d6ab9013               5  high_priority_refresh   
7  content_b511d4bc4ad2               5  high_priority_refresh   
8  content_d17681677e69               5  high_priority_refresh   
9  content_a7427266c305               5  high_priority_refresh   

               action  impressions_90d   ctr  days_since_last_update  
0  Review and Refresh           517715  0.14                     104  
1  Review and Refresh           443434  0.21                     104  
2  Review and Refresh           3

##3. Top-20 Review

**The top 20 pages were all assigned the "Review and Refresh" action because they satisfied all three baseline conditions: high impressions, low CTR, and stale content. The reason code for all of them is high_priority_refresh.**

**These pages have strong visibility but relatively low click-through rates and have not been updated recently, making them good candidates for content refresh.**

The confidence in these recommendations is high because they satisfy every condition in the baseline rule. However, the recommendations could be wrong if:

*   the page was recently updated outside the available data,
*   the low CTR is caused by search result features rather than page quality,
*   the page targets keywords that naturally receive low CTR,
*  seasonal trends temporarily affected impressions or clicks.





In [ ]:
top20 = ranked.head(20)

top20[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "days_since_last_update"
    ]
]

,content_id,baseline_score,reason_code,action,impressions_90d,ctr,days_since_last_update
0,content_5fe46e04994d,5,high_priority_refresh,Review and Refresh,517715,0.14,104
1,content_2dba2b1f9536,5,high_priority_refresh,Review and Refresh,443434,0.21,104
2,content_cb112fce36be,5,high_priority_refresh,Review and Refresh,309910,0.16,104
3,content_36ff89c8214e,5,high_priority_refresh,Review and Refresh,295097,0.05,104
4,content_b28d1efd668f,5,high_priority_refresh,Review and Refresh,286608,0.06,104
5,content_813e88069237,5,high_priority_refresh,Review and Refresh,233561,0.06,104
6,content_c8e9d6ab9013,5,high_priority_refresh,Review and Refresh,208678,0.00,104
7,content_b511d4bc4ad2,5,high_priority_refresh,Review and Refresh,205915,0.14,104
8,content_d17681677e69,5,high_priority_refresh,Review and Refresh,201584,0.24,104
9,content_a7427266c305,5,high_priority_refresh,Review and Refresh,201111,0.11,104


## 4. Weak picks + leakage check

**The weakest recommendations would be pages that satisfy the thresholds but have low CTR for reasons unrelated to content quality, such as search intent, SERP features, or seasonal effects. These pages may not benefit from a content refresh.**

No leakage was introduced in this baseline. The rule only uses features available before making the decision:

* impressions_90d
* ctr
* days_since_last_update

*The rule does not use trend_direction or trend_pct, since these are derived from future outcomes and would introduce label leakage. No product flags or future-window information were used.*

In [ ]:
# Verify that no label-derived columns were used

features_used = [
    "impressions_90d",
    "ctr",
    "days_since_last_update"
]

print("Features used:", features_used)

for col in ["trend_direction", "trend_pct"]:
    print(f"{col} used?", col in features_used)

Features used: ['impressions_90d', 'ctr', 'days_since_last_update']
trend_direction used? False
trend_pct used? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.